In [1]:
import sys
sys.path.insert(0, "pythonlib")
import pandas as pd
from enginewash import WashCalculator, WashConfig, FlightRecord, FlightPhase, MaintenanceRecord, EGTHDM

ENGINES = ["657287", "804853"]
START, END = "2025-05-01", "2026-05-01"

maintenance = pd.read_parquet("https://storage.yandexcloud.net/ecm-data/ecmapp.maintenance_20260610.parquet")
takeoff = pd.read_parquet("https://storage.yandexcloud.net/ecm-data/s7.b737_takeoff_20260222-merged.parquet")

takeoff = takeoff[["engine_id", "flight_datetime", "egthdm"]].dropna()
takeoff["engine_id"] = takeoff["engine_id"].astype(int).astype(str)
takeoff = takeoff[
    takeoff["engine_id"].isin(ENGINES)
    & (takeoff["flight_datetime"] >= START)
    & (takeoff["flight_datetime"] <= END)
]

flights = [
    FlightRecord(
        engine_id=row.engine_id,
        flight_datetime=row.flight_datetime,
        parameter_name="EGTHDM",
        flight_phase=FlightPhase.TAKEOFF,
        float_value=row.egthdm,
    )
    for row in takeoff.itertuples()
]

wash = maintenance[
    maintenance["ata_code"].isin(['330', '340', '331'])
    & ~maintenance["deleted"]
    & maintenance["engine_id"].isin(ENGINES) 
    & (maintenance["maint_datetime"] >= START)
    & (maintenance["maint_datetime"] <= END)
]

wash = wash.sort_values("mutation_time").drop_duplicates(["engine_id", "maint_datetime"], keep="first")

maintenance_records = [
    MaintenanceRecord(
        engine_id=row.engine_id,
        maint_datetime=pd.to_datetime(row.maint_datetime),
        ata_code=row.ata_code
    )
    for row in wash.itertuples()
]

calc = WashCalculator()
summaries = calc.process(flights=flights, maintenances=maintenance_records, utilizations=[], parameter=EGTHDM)

result = pd.DataFrame([
    {
        "engine_id": ev.engine_id,
        "event_index": ev.event_index,
        "maint_date": ev.maint_datetime,
        "ata_code": ev.ata_code,
        "mean_before": round(ev.mean_before, 2),
        "mean_after": round(ev.mean_after, 2),
        "delta": round(ev.delta, 2),
        "loss_of_efficiency": ev.time_loss_of_efficiency,
    }
    for s in summaries for ev in s.results
])
result

,engine_id,event_index,maint_date,ata_code,mean_before,mean_after,delta,loss_of_efficiency
0,657287,1,2025-09-19,330,33.00,37.52,4.51,2025-12-07 16:43:45
1,657287,2,2025-12-22,340,37.17,45.02,7.84,NaT
2,804853,1,2025-06-29,331,42.03,49.08,7.05,NaT
3,804853,2,2025-11-02,330,50.61,54.99,4.39,2025-11-24 18:31:41
4,804853,3,2025-12-12,331,48.05,50.97,2.91,2026-02-03 19:41:24
